# 1. Data Loading

In [2]:
import sys
!{sys.executable} -m pip install imbalanced-learn optuna


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score,f1_score, matthews_corrcoef, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score, StratifiedKFold


from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN, SMOTETomek

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42


In [4]:
X_train = pd.read_csv("../Train_test_split/X_train.csv")
X_test  = pd.read_csv("../Train_test_split/X_test.csv")
y_train = pd.read_csv("../Train_test_split/y_train.csv")
y_test  = pd.read_csv("../Train_test_split/y_test.csv")

print(f'Train set: {X_train.shape[0]:,} samples | {X_train.shape[1]} features')
print(f'Test  set: {X_test.shape[0]:,} samples')
print(f'\nFeatures: {list(X_train.columns)}')
X_train.head()

Train set: 40,000 samples | 19 features
Test  set: 10,000 samples

Features: ['age', 'annual_income', 'months_active', 'avg_monthly_spend', 'purchase_frequency', 'avg_order_value', 'discount_usage_rate', 'return_rate', 'browsing_time_minutes', 'support_interactions', 'spend_per_browse', 'monthly_value', 'net_purchase_rate', 'loyalty_score', 'clv_proxy', 'payment_method_UPI', 'payment_method_Wallet', 'region_Semi-Urban', 'region_Urban']


,age,annual_income,months_active,avg_monthly_spend,purchase_frequency,avg_order_value,discount_usage_rate,return_rate,browsing_time_minutes,support_interactions,spend_per_browse,monthly_value,net_purchase_rate,loyalty_score,clv_proxy,payment_method_UPI,payment_method_Wallet,region_Semi-Urban,region_Urban
0,-0.457168,-0.246994,0.988371,-1.342790,0.304190,-1.128648,0.044661,2.088832,1.603505,-1.348632,-1.135439,-1.229367,-2.088832,1.067645,-0.849767,False,False,False,False
1,-0.915153,-0.623346,-1.656322,0.294737,-0.327284,0.146140,-0.337552,-1.139958,-0.533765,-0.411962,0.281601,0.339950,1.139958,-1.115634,-1.140345,False,False,True,False
2,0.785933,0.941846,-1.608236,-0.454365,-0.171645,-0.480279,0.024451,-0.311039,0.007130,-1.348632,-0.582289,-0.362537,0.311039,-1.079823,-1.151153,True,False,False,False
3,0.262522,0.109167,1.084542,-1.218122,-0.982540,-0.646064,-1.132617,-0.501766,0.037160,1.461380,-0.966319,-1.247255,0.501766,-0.426608,-0.524638,False,False,False,True
4,-1.242285,-0.899839,1.132627,-1.104892,1.384401,-1.124646,0.563397,-0.731998,2.464325,-1.348632,-1.118972,-0.936819,0.731998,2.499322,-0.352617,False,True,True,False


# 3.Imbalance Mitigation


In [5]:
def evaluate_model(model, X_tr, y_tr, X_te, y_te, name):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    return {
        'Strategy': name,
        'Accuracy':  round(accuracy_score(y_te, y_pred), 4),
        'Precision': round(precision_score(y_te, y_pred, average='macro', zero_division=0), 4),
        'Recall':    round(recall_score(y_te, y_pred, average='macro', zero_division=0), 4),
        'F1 (macro)':round(f1_score(y_te, y_pred, average='macro', zero_division=0), 4),
        'MCC':       round(matthews_corrcoef(y_te, y_pred), 4)
    }

base_gb = GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)


strategies = [
    ('No Resampling (Baseline)',
     GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)),

    ('RandomOverSampler',
     ImbPipeline([
         ('sampler', RandomOverSampler(random_state=RANDOM_STATE)),
         ('clf',     GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE))
     ])),

    ('SMOTE',
     ImbPipeline([
         ('sampler', SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
         ('clf',     GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE))
     ])),

    ('RandomUnderSampler',
     ImbPipeline([
         ('sampler', RandomUnderSampler(random_state=RANDOM_STATE)),
         ('clf',     GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE))
     ])),

    ('SMOTEENN',
     ImbPipeline([
         ('sampler', SMOTEENN(random_state=RANDOM_STATE)),
         ('clf',     GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE))
     ])),

    ('SMOTETomek',
     ImbPipeline([
         ('sampler', SMOTETomek(random_state=RANDOM_STATE)),
         ('clf',     GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE))
     ])),
]

results = []
for name, pipeline in strategies:
    print(f'Training: {name}...', end=' ')
    res = evaluate_model(pipeline, X_train, y_train, X_test, y_test, name)
    results.append(res)
    print(f'F1={res["F1 (macro)"]:.4f} | MCC={res["MCC"]:.4f}')


print('Training: class_weight=balanced...', end=' ')
gb_balanced = GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)

from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight('balanced', y_train)
gb_balanced.fit(X_train, y_train, sample_weight=sample_weights)
y_pred_bal = gb_balanced.predict(X_test)
res_bal = {
    'Strategy': 'class_weight=balanced',
    'Accuracy':  round(accuracy_score(y_test, y_pred_bal), 4),
    'Precision': round(precision_score(y_test, y_pred_bal, average='macro', zero_division=0), 4),
    'Recall':    round(recall_score(y_test, y_pred_bal, average='macro', zero_division=0), 4),
    'F1 (macro)':round(f1_score(y_test, y_pred_bal, average='macro', zero_division=0), 4),
    'MCC':       round(matthews_corrcoef(y_test, y_pred_bal), 4)
}
results.append(res_bal)
print(f'F1={res_bal["F1 (macro)"]:.4f} | MCC={res_bal["MCC"]:.4f}')

results_df = pd.DataFrame(results)
print('\nDone!')

#El mejor resultado es el de No Resampling, GradientBoosting funciona muy bien con datasets
#Medianamente desbalanceados, por ello al forzar un balanceo antinatura, los resultados pueden
#Empeorar. Con RandomOverSampler al copiar columnas iguales constantemente existe grandes posibilidades
#De sobre ajuste. SMOTE lo que hace es fijarse en vecinos y crear datos nuevos en base a ello, por eso 
#Existen las posibilidades de creacion de data que pertenezca a otra clase. En RandomUnderSampling
#Puede eliminar datos reales e importantes para el modelo para aprender mejor.Smoteen infla la clase 
#Mayoritaria y luegi hace una limpieza, tanto de puntos reales como sinteticos, eso hace que puede
#Que puntos diferentes a sus vecinos pero reales desaparezcan.SmoteTomek, hace tambien la inflacion de
#clases, lo que hace es busca los links tomek, que son links que por lo general suponen que estan en 
#Una zona de confusion, entonces lo que hace es eliminar ambos puntos o unicamente los de la clase 
#Mayoritaria, este tipo de mitigation es mas precisa, mas analitica por lo que ni elimina agresivamente
#Ni se inventa unicamente datos, se basa en la confusion para eliminar





#GradientBoosting funciona bien con datasets medianamente desbalanceados, principalmente porque 
#Es un modelo de secuencialidad, esto que significa? que va creando arbolitos poco a poco, empieza
#Por los casos mas faciles por asi decirlo, por ello empieza con la parte que tiene mas % haciendo 
#Que salgan errores no en el modelo en general ni en la parte de alto porcentaje, si no que en la parte
#minoritaria, haciendo que el modelo diga "Anda hay un error aqui, me voy a fijar mas", por lo que en 
#La siguiente iteracion se fija un poquito mas en la parte minoritaria y así consecutivamente.
#Si fuese un caso de desbalanceo extremo, el modelo trataria la clase minoritaria como outliers

Training: No Resampling (Baseline)... 

KeyboardInterrupt: 